# Deepfake Detection — Dataset Setup & Data Loading

Single working notebook for the deepfake image detection experiment (all milestones build on this file).

## Milestone 1 — Dataset setup

Goal: load the dataset, confirm class balance, inspect sample images, and record the split sizes.

Dataset: [TheKernel01/140k-Real-and-Fake-Faces](https://huggingface.co/datasets/TheKernel01/140k-Real-and-Fake-Faces) (HuggingFace mirror).

- 70k real faces from Flickr-Faces-HQ (FFHQ)
- 70k fake faces generated by StyleGAN
- Pre-split: train / validation / test
- All images resized to 256x256

Why this dataset: it is balanced (no class imbalance), self-contained (real photos vs StyleGAN fakes), and there is a published benchmark showing fine-tuned ResNet50 reaching ~99.6% test accuracy, giving us a sanity-check target.

## Milestone 2 — Data loading

Goal: wrap the HuggingFace dataset into a PyTorch `Dataset` and feed it through a `DataLoader`, applying the designed transforms:

- `Resize(224)` — models are trained for 224x224 input
- `ToTensor()` — PIL image `[0,255]` HWC to torch tensor `[0,1]` CHW
- `Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])` — ImageNet stats

## Setup — Google Colab

In [ ]:
!pip install -q datasets torch torchvision

## Load the dataset

`load_dataset` downloads (and caches) the parquet shards on first call. The cache persists only for the Colab session.

In [ ]:
from datasets import load_dataset

ds = load_dataset("TheKernel01/140k-Real-and-Fake-Faces")
print(ds)

## Class balance

Confirm each split is balanced. For a binary classification task, balanced classes mean accuracy is a meaningful primary metric. If the data were imbalanced, a model predicting the majority class would score high accuracy while being useless.

In [ ]:
from collections import Counter

for split in ds:
    labels = ds[split]["label"]
    counts = Counter(labels)
    total = len(labels)
    print(f"{split:12s}: total={total:6d}, real={counts[0]:6d}, fake={counts[1]:6d}")
    print(f"{'':12s}  real={counts[0]/total:.2%}, fake={counts[1]/total:.2%}")

## Inspect images

The `datasets` library lazily decodes images (decoded=True by default for these columns), so each access to `dataset[i]["image"]` loads the JPEG into a PIL Image. We only index a few samples here to keep it cheap.

Labels: `0 = real`, `1 = fake`.

In [ ]:
from PIL import Image

for i in range(3):
    img = ds["train"][i]["image"]
    label = ds["train"][i]["label"]
    print(f"idx={i}: size={img.size}, mode={img.mode}, label={label} ({'real' if label==0 else 'fake'})")

In [ ]:
import matplotlib.pyplot as plt

real_indices = [i for i, l in enumerate(ds["train"]["label"]) if l == 0][:3]
fake_indices = [i for i, l in enumerate(ds["train"]["label"]) if l == 1][:3]

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for col, idx in enumerate(real_indices):
    axes[0, col].imshow(ds["train"][idx]["image"])
    axes[0, col].set_title("real")
    axes[0, col].axis("off")
for col, idx in enumerate(fake_indices):
    axes[1, col].imshow(ds["train"][idx]["image"])
    axes[1, col].set_title("fake")
    axes[1, col].axis("off")
plt.tight_layout()
plt.show()

## Transforms

Two separate pipelines, because train and validation have different needs:

- **Train**: `RandomHorizontalFlip()` adds a free, label-preserving augmentation. Flipping a real face still looks like a real face and flipping a fake still looks fake, so it never corrupts the label.
- **Validation**: no augmentation. Validation/test must measure the model on untouched data so the metric stays meaningful and comparable.

Both end with Resize -> ToTensor -> Normalize. ImageNet mean/std are used because the pretrained backbone (M3) expects inputs normalized exactly this way to match its training distribution.

In [ ]:
from torchvision import transforms

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

print("train transform:", train_transform)
print("eval transform:", eval_transform)

## Wrap the HF split into a torch Dataset

The `hf_split["image"]` accessor already hands back a decoded PIL Image, so `__getitem__` only needs to locate the sample, run the transform, and return `(image, label)`.

A single generic class serves all splits (train / validation / test); the transform is injected so the same class handles both augmented and non-augmented paths.

In [ ]:
from torch.utils.data import Dataset


class DeepfakeDataset(Dataset):
    """Wraps one HuggingFace split (train/val/test) into a torch Dataset."""
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        image = self.hf_split[idx]["image"]  # decoded PIL Image
        label = self.hf_split[idx]["label"]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


train_ds = DeepfakeDataset(ds["train"], transform=train_transform)
val_ds   = DeepfakeDataset(ds["validation"], transform=eval_transform)
test_ds  = DeepfakeDataset(ds["test"], transform=eval_transform)

print(f"train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

## DataLoaders

- `shuffle=True` for train: each epoch presents a different sample order so the optimizer never memorizes a fixed sequence.
- `shuffle=False` for val: order is irrelevant for scoring, and a fixed order keeps validation reproducible.
- `num_workers=2` prefetches batches in parallel to keep the GPU fed.

In [ ]:
from torch.utils.data import DataLoader

batch_size = 64
num_workers = 2

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"train batches/epoch: {len(train_loader)}")
print(f"val batches/epoch:   {len(val_loader)}")
print(f"test batches/epoch:  {len(test_loader)}")

## Sanity check — one batch

Verify the shapes and pixel statistics before training.

After `ToTensor()` + `Normalize(imagenet stats)`, the batch mean should be near 0 and std near 1. That is expected: normalization subtracts the ImageNet mean and divides by the ImageNet std, so per-channel values are recentered and rescaled onto an approximately unit-standard-deviation scale.

In [ ]:
import torch

images, labels = next(iter(train_loader))

print("images.shape:", images.shape)   # expect (batch, 3, 224, 224)
print("labels.shape:", labels.shape)   # expect (batch,)
print("label values:", torch.unique(labels))
print("batch mean:", images.mean(dim=(0, 2, 3)).tolist())
print("batch std:", images.std(dim=(0, 2, 3)).tolist())

## Milestone 3 — Model & Training

Build the detector as a pretrained ImageNet ResNet50 with a 2-class head, then train it.

### Model
- Backbone: `resnet50(weights=IMAGENET1K_V1)` — pretrained on ImageNet.
- Head: replace `model.fc` with `Linear(2048, 2)` (2048 features -> 2 logits for real/fake).

Why pretrained + new head only: the convolutional stack learns a reusable low/mid-level visual vocabulary (edges, textures, gradients) that transfers to face/artifact classification, so we keep it and only swap the ImageNet-specific decision layer (1000 classes -> 2).

### Staged training (progressive unfreezing)
We do not jump straight to full fine-tuning. We first train only the new head, then unfreeze the last stage(s) at a smaller learning rate. This preserves the useful pretrained representation while letting it adapt to deepfake evidence, and limits overfitting during the early stage.

Different learning rates reflect different prior knowledge:
- New `fc` head: starts random -> higher LR (`1e-3`)
- Pretrained backbone: already useful -> lower LR (`1e-4`)



## Device

Move the model and batches to GPU when available.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Build the model

`resnet50(weights=...)` currently returns a model whose final layer outputs 1000 logits (ImageNet). We replace that layer with a 2-output head.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

num_features = model.fc.in_features  # 2048
model.fc = nn.Linear(num_features, 2)

model = model.to(device)
print("model fc head:", model.fc)
print("num trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## Parameter groups with different learning rates

Split parameters into the new head (random init) and the pretrained backbone. The optimizer applies a higher LR to the head and a lower LR to the backbone.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Distinct parameter groups so the head trains faster than the backbone
backbone_params = [p for name, p in model.named_parameters() if "fc" not in name]
head_params     = [p for name, p in model.named_parameters() if "fc" in name]

optimizer = optim.Adam([
    {"params": backbone_params, "lr": 1e-4},
    {"params": head_params,     "lr": 1e-3},
])
criterion = nn.CrossEntropyLoss()

print(f"backbone params: {len(backbone_params)}, head params: {len(head_params)}")

## Stage 1 — train only the new head (backbone frozen)

Freeze the backbone initially. This is cheap and avoids destroying the pretrained representation before the head has learned to map 2048 features to real/fake.

`loss.backward()` then `optimizer.step()` performs:

$$\theta_{t+1} = \theta_t - \eta \nabla_\theta L_{\text{batch}}$$

Only the head's parameters get non-zero gradients while `requires_grad=False` on the backbone.

In [ ]:
def set_requires_grad(model, requires_grad):
    for param in model.parameters():
        param.requires_grad = requires_grad


# Stage 1: freeze everything except the new head
set_requires_grad(model, False)
for name, param in model.named_parameters():
    if "fc" in name:
        param.requires_grad = True

print("Stage 1 trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## Training loop

One forward pass, loss, backward, and a step per batch; repeat per epoch. `model.train()` enables training-mode behaviors (batch norm updates, dropout).

We also track running loss to watch the trend.

In [ ]:
from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0
    for images, labels in tqdm(loader, desc="train"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)                 # (batch, 2)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        total += len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="val"):
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            total += len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
    return total_loss / total, correct / total

In [ ]:
epochs_stage1 = 3

for epoch in range(1, epochs_stage1 + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)
    print(f"stage1 epoch {epoch}: train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

## Stage 2 — fine-tune the pretrained layers

After the head has learned to map features to real/fake, unfreeze the backbone and continue at the lower (backbone) learning rate. Smaller steps protect the learned representation from being destroyed before the new task can guide it.

We only run a couple of epochs here to validate the pipeline before committing to a longer run.

In [ ]:
# Unfreeze the whole network for fine-tuning
set_requires_grad(model, True)

# Recreate optimizer (backbone params now require grad) with the same differential LRs
backbone_params = [p for name, p in model.named_parameters() if "fc" not in name]
head_params     = [p for name, p in model.named_parameters() if "fc" in name]
optimizer = optim.Adam([
    {"params": backbone_params, "lr": 1e-4},
    {"params": head_params,     "lr": 1e-3},
])

epochs_stage2 = 2

for epoch in range(1, epochs_stage2 + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)
    print(f"stage2 epoch {epoch}: train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

## Save the checkpoint

Persist the best trained weights so we can load them for evaluation (M4) and the Streamlit app (M5). We save only the `state_dict` (weights), not the whole model object, which is the portable convention.

We save to Google Drive so the weights survive the ephemeral Colab runtime and we never have to retrain after restarting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = "/content/drive/MyDrive/deepfake_resnet50"
os.makedirs(save_dir, exist_ok=True)

checkpoint_path = f"{save_dir}/deepfake_resnet50_v1.pt"
torch.save(model.state_dict(), checkpoint_path)
print(f"saved checkpoint: {checkpoint_path}")

## Milestone 4 — Evaluation

Measure the trained model on the held-out **test** split (20k images never seen during training). Accuracy alone is insufficient for a detector, so we report:

- Accuracy
- Precision / Recall / F1 (macro)
- Confusion matrix
- ROC curve and AUC

We compute these from the **class probabilities** (softmax over the 2 logits), which lets us trace out the full ROC curve by sweeping the decision threshold.

## Reload the trained checkpoint

If this notebook was restarted, rebuild the model and load the saved weights before evaluating. The architecture must match exactly (same head size).

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Mount Drive to reach the saved checkpoint
from google.colab import drive
drive.mount('/content/drive')

checkpoint_path = "/content/drive/MyDrive/deepfake_resnet50/deepfake_resnet50_v1.pt"

# Rebuild the exact same architecture
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)

# Load the saved weights
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model = model.to(device)
model.eval()
print(f"model loaded from {checkpoint_path}")
# Ensure the loaders from M2 still exist; if not, rebuild them below
try:
    test_loader
except NameError:
    from datasets import load_dataset
    from torchvision import transforms
    from torch.utils.data import Dataset, DataLoader

    ds = load_dataset("TheKernel01/140k-Real-and-Fake-Faces")
    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    class DeepfakeDataset(Dataset):
        def __init__(self, hf_split, transform=None):
            self.hf_split = hf_split
            self.transform = transform
        def __len__(self):
            return len(self.hf_split)
        def __getitem__(self, idx):
            image = self.hf_split[idx]["image"]
            label = self.hf_split[idx]["label"]
            if self.transform is not None:
                image = self.transform(image)
            return image, label

    test_ds = DeepfakeDataset(ds["test"], transform=eval_transform)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
    print("rebuilt test_loader")

print("test batches:", len(test_loader))

## Gather predictions and probabilities

Run the model over the test set once, collecting softmax probabilities and true labels. `torch.no_grad()` disables autograd to save memory since we only forward.

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm

softmax = nn.Softmax(dim=1)

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="predicting"):
        images = images.to(device)
        logits = model(images)
        probs = softmax(logits)          # (batch, 2) rows sum to 1
        all_probs.append(probs[:, 1].cpu().numpy())  # P(fake)
        all_labels.append(labels.numpy())

y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)
y_pred = (y_prob >= 0.5).astype(int)

print("samples:", len(y_true))
print("predicted fakes:", y_pred.sum(), "|  true fakes:", y_true.sum())

## Classification report

Precision = fraction of predicted fake that are truly fake; Recall = fraction of true fakes we caught. For a detector, recall on fakes matters (missed fakes are dangerous), and we want precision high too so real faces aren't mislabelled as fake.

Macro F1 averages both classes equally.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=["real", "fake"]))

## Confusion matrix

A 2x2 grid of predicted vs true. The two error cells tell us the failure mode:
- top-right: real faces wrongly declared fake (false alarm)
- bottom-left: fake faces not detected (missed fake)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["real", "fake"])
disp.plot(cmap="Blues")
plt.title("Confusion matrix (test)")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"false alarm rate (real->fake): {fp/(fp+tn):.4f}")
print(f"missed fake rate (fake missed): {fn/(fn+tp):.4f}")

## ROC curve and AUC

ROC sweeps the decision threshold over all possible values and plots the true-positive rate vs false-positive rate. AUC is the chance a random fake scores higher than a random real — 1.0 is perfect, 0.5 is chance. It is threshold-independent, so it characterizes the model's ranking quality overall, not just at one threshold.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="chance")
plt.xlabel("False positive rate (real -> fake)")
plt.ylabel("True positive rate (fake found)")
plt.title("ROC curve (test)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC: {auc:.4f}")

## Operating point note

The default threshold is 0.5, but a detector can tune it. If missing fakes is the worst failure, lower the threshold to catch more fakes (raises recall at the cost of precision, i.e. more false alarms on real faces). The ROC curve shows this tradeoff at a glance.

## M7 — Multi-generator retraining (fix shortcut learning)

Quick recap. The M3/M4 model scored 99.4% on the 140k benchmark. It was too good:
in that dataset every real face is FFHQ and every fake face is StyleGAN2.

Real-world check with AI images from generators we never trained on:
- Gemini boy image          -> predicted REAL (99.5%)
- ChatGPT generated faces   -> predicted REAL (100%)

The model learned "StyleGAN vs FFHQ", not "AI vs real". Fix = multi-generator training:
add fakes from many methods so the model learns what makes an image look machine-made.

The plan below was adapted twice as we verified the data:
1. `pujanpaudel/deepfake_face_classification` (DF40, 40 techniques) — its train.rar only
   contained 3,415 fake faces (no real folder, no per-technique names). Kept ONLY as a
   face anchor: fakes join the training mix, `test` stays as a held-out face check.
2. Main new dataset: `Rajarshi-Roy-research/Defactify_Image_Dataset` (MS COCOAI, 2026) —
   real photos + 5 modern text-to-image generators (SD2.1, SDXL, SD3, DALL-E 3,
   MidJourney v6) of the SAME captions, with binary (Label_A) and per-model (Label_B)
   labels. This directly teaches diffusion artifacts.

Remaining plan: fine-tune v1 -> per-generator eval -> 140k/DF40 face checks ->
real-world Gemini/ChatGPT retest -> save v2.

In [ ]:
import os
import urllib.request

DATA_DIR = "/content/df40"
os.makedirs(DATA_DIR, exist_ok=True)

BASE = "https://huggingface.co/datasets/pujanpaudel/deepfake_face_classification/resolve/main"
files = {name: f"{BASE}/{name}?download=true"
         for name in ["train.rar", "val.zip", "test.zip"]}

for name, url in files.items():
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest):
        print(f"already downloaded: {name} ({os.path.getsize(dest)/1e9:.2f} GB)")
        continue
    print(f"downloading {name} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"  done: {name} ({os.path.getsize(dest)/1e9:.2f} GB)")

In [ ]:
!apt-get install -y -q unrar

In [ ]:
import os
import subprocess

EXTRACT_DIR = "/content/df40_extract"
os.makedirs(EXTRACT_DIR, exist_ok=True)

for name, out in [("train.rar", "train"), ("val.zip", "val"), ("test.zip", "test")]:
    dest = os.path.join(DATA_DIR, name)
    out_dir = os.path.join(EXTRACT_DIR, out)
    if not os.path.exists(dest):
        raise FileNotFoundError(f"download {name} first")
    if os.path.exists(out_dir) and os.listdir(out_dir):
        print(f"already extracted: {out}")
        continue
    os.makedirs(out_dir, exist_ok=True)
    print(f"extracting {name} ...")
    if name.endswith(".zip"):
        subprocess.run(["unzip", "-q", "-o", dest, "-d", out_dir], check=True)
    else:
        subprocess.run(["unrar", "x", "-y", "-o+", dest, f"{out_dir}/"], check=True)
    print(f"  done: {out}")

print("extracted roots:", sorted(os.listdir(EXTRACT_DIR)))

In [ ]:
import os

def show_tree(root, max_depth=3):
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth >= max_depth:
            dirnames[:] = []
            continue
        rel = dirpath[len(root):]
        print(f"{'  ' * depth}{os.path.basename(rel) if rel else '.'}/")
        for d in sorted(dirnames)[:10]:
            print(f"{'  ' * depth}  {d}/")


for sub in sorted(os.listdir(EXTRACT_DIR)):
    print(f"== {sub}")
    show_tree(os.path.join(EXTRACT_DIR, sub))

### M7 data — what we keep and what we add

The DF40 mirror we downloaded was deficient: `train.rar` held only 3,415 fake faces
(no `real` folder, no per-technique names). So DF40 is kept ONLY as a face anchor:
its fakes go into the training mix and `test` stays as a held-out face check.

Primary new dataset: `Rajarshi-Roy-research/Defactify_Image_Dataset` (MS COCOAI, 2026).
- 96,000 images: each COCO caption has the REAL photo plus 5 AI versions
- generators: Stable Diffusion 2.1, SDXL, SD3, DALL-E 3, MidJourney v6
- `Label_A`: 0=real, 1=AI      `Label_B`: 0=real, 1..5 = model
- splits: train 42,000 / val 9,000 / test 45,000 (parquet, load_dataset)

Why this kills our failure mode: v1 only ever saw StyleGAN fakes, so diffusion faces
(Gemini, ChatGPT) looked real. On Defactify the model must separate a real photo from
the SAME scene generated by 5 diffusion models — it has to learn generation artifacts,
not a "scenes vs faces" content shortcut. DF40 + FFHQ keep it a face detector.

In [ ]:
from collections import Counter
from datasets import load_dataset

defactify = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset")
print("splits:", list(defactify))

gen_names = {0: "real", 1: "SD2.1", 2: "SDXL", 3: "SD3", 4: "DALL-E 3", 5: "MidJourney v6"}

for split in defactify:
    a = defactify[split]["Label_A"]
    b = defactify[split]["Label_B"]
    print(f"{split.capitalize():12s} rows={len(a):6d}  A: {dict(Counter(a))}   B: {dict(Counter(b))}")

print("\nsanity rows (train):")
for i in range(6):
    r = defactify["train"][i]
    print(f"  [{i}] A={r['Label_A']} B={r['Label_B']} ({gen_names.get(r['Label_B'], '?')})  caption={r['Caption'][:42]!r}")

In [ ]:
import glob
import os
from PIL import Image
from torch.utils.data import Dataset

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


class FaceFolderDataset(Dataset):
    """Labels by folder: images under a `fake` dir are class 1, under `real` are class 0."""
    def __init__(self, root, transform=None):
        self.items = []  # (path, label)
        for path in sorted(glob.glob(os.path.join(root, "**", "*"), recursive=True)):
            if os.path.splitext(path)[1].lower() not in IMG_EXTS:
                continue
            parts = [p.lower() for p in path.split(os.sep)]
            if "fake" in parts:
                label = 1
            elif "real" in parts:
                label = 0
            else:
                continue
            self.items.append((path, label))
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def find_class_root(start):
    for dirpath, dirnames, _ in os.walk(start):
        lower = [d.lower() for d in dirnames]
        if "fake" in lower or "real" in lower:
            return dirpath
    return None


fake_root = find_class_root(os.path.join(EXTRACT_DIR, "train"))  # train.rar held only fakes
df40_fake_train = FaceFolderDataset(fake_root, transform=train_transform)
df40_test_ds    = FaceFolderDataset(find_class_root(os.path.join(EXTRACT_DIR, "test")), transform=eval_transform)
print(f"DF40 face fakes (training anchor): {len(df40_fake_train)}")
print(f"DF40 face holdout set (test):      {len(df40_test_ds)}")

In [ ]:
import random
import torch
from torch.utils.data import Dataset, ConcatDataset


class HFSubset(Dataset):
    """A deterministic subset of a HuggingFace split with our transform applied."""
    def __init__(self, hf, indices, image_key="Image", label_key="Label_A", transform=None):
        self.hf = hf
        self.indices = indices
        self.image_key = image_key
        self.label_key = label_key
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row = self.hf[self.indices[idx]]
        image = row[self.image_key]
        if self.transform is not None:
            image = self.transform(image)
        return image, row[self.label_key]


# --- Defactify: equal-size real vs fake, fakes stratified across all 5 generators ---
df_train = defactify["train"]
a = df_train["Label_A"]
b = df_train["Label_B"]

real_idx_all = [i for i, x in enumerate(a) if x == 0]
fake_by_gen = {}
for i, x in enumerate(a):
    if x == 1:
        fake_by_gen.setdefault(b[i], []).append(i)

N_PER_GEN = 1000
fake_sel = []
for g in sorted(fake_by_gen):
    fake_sel += random.Random(42).sample(fake_by_gen[g], min(N_PER_GEN, len(fake_by_gen[g])))
real_sel = random.Random(42).sample(real_idx_all, len(fake_sel))

defactify_real_ds = HFSubset(df_train, real_sel, transform=train_transform)
defactify_fake_ds = HFSubset(df_train, fake_sel, transform=train_transform)

# --- FFHQ real faces, same count as the DF40 face fakes (keeps it a face detector) ---
ffhq_real_indices = [i for i, l in enumerate(ds["train"]["label"]) if l == 0]
ffhq_sel = random.Random(7).sample(ffhq_real_indices, len(df40_fake_train))
ffhq_real_ds = HFSubset(ds["train"], ffhq_sel, image_key="image", label_key="label", transform=train_transform)

m7_train_ds = ConcatDataset([defactify_real_ds, ffhq_real_ds, defactify_fake_ds, df40_fake_train])

real_n = len(defactify_real_ds) + len(ffhq_real_ds)
fake_n = len(defactify_fake_ds) + len(df40_fake_train)
print(f"M7 train: total={len(m7_train_ds)}  real={real_n} ({len(defactify_real_ds)} defactify + {len(ffhq_real_ds)} ffhq)  fake={fake_n} ({len(defactify_fake_ds)} defactify + {len(df40_fake_train)} df40)")

In [ ]:
import matplotlib.pyplot as plt

rows = [
    ("real (Defactify/COCO)", defactify_real_ds),
    ("fake (Defactify diffusion)", defactify_fake_ds),
    ("fake (DF40 faces)", df40_fake_train),
]
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for r, (title, dset) in enumerate(rows):
    for c in range(6):
        img, _ = dset[c]
        img = img * torch.tensor(imagenet_std).view(3, 1, 1) + torch.tensor(imagenet_mean).view(3, 1, 1)
        axes[r][c].imshow(img.permute(1, 2, 0).clip(0, 1))
        axes[r][c].axis("off")
    axes[r][0].set_ylabel(title, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import DataLoader

val_ds_def  = HFSubset(defactify["validation"], list(range(len(defactify["validation"]))), transform=eval_transform)
test_ds_def = HFSubset(defactify["test"],      list(range(len(defactify["test"]))),      transform=eval_transform)

m7_train_loader = DataLoader(m7_train_ds,  batch_size=batch_size, shuffle=True,  num_workers=num_workers)
val_loader       = DataLoader(val_ds_def,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
def_test_loader  = DataLoader(test_ds_def,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
df40_test_loader = DataLoader(df40_test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"train batches: {len(m7_train_loader)}, val: {len(val_loader)}, defactify-test: {len(def_test_loader)}, df40-test: {len(df40_test_loader)}")

### M7 stage 3 fine-tune

Start from the v1 checkpoint (already great on StyleGAN) and fine-tune on the balanced
M7 set: real = Defactify COCO reals + FFHQ faces; fake = 5 Defactify diffusion
generators + DF40 face fakes. Lower learning rates than stage 2 so the model refines
what it already knows instead of thrashing. All layers trainable.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_v1 = "/content/drive/MyDrive/deepfake_resnet50/deepfake_resnet50_v1.pt"

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load(checkpoint_v1, map_location=device))
model = model.to(device)
set_requires_grad(model, True)

backbone_params = [p for name, p in model.named_parameters() if "fc" not in name]
head_params     = [p for name, p in model.named_parameters() if "fc" in name]
optimizer = optim.Adam([
    {"params": backbone_params, "lr": 5e-5},
    {"params": head_params,     "lr": 5e-4},
])
criterion = nn.CrossEntropyLoss()

epochs_stage3 = 3
best_val_acc = 0.0
for epoch in range(1, epochs_stage3 + 1):
    tr_loss, tr_acc = train_one_epoch(model, m7_train_loader, optimizer, criterion, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)
    print(f"epoch {epoch}/{epochs_stage3} -> train loss={tr_loss:.4f} acc={tr_acc:.4f} | val loss={va_loss:.4f} acc={va_acc:.4f}")
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), "/content/m7_best.pt")
        print(f"  saved new best (val acc {va_acc:.4f})")

model.load_state_dict(torch.load("/content/m7_best.pt", map_location=device))
print(f"best val accuracy: {best_val_acc:.4f}")

### Evaluate

1. **Per generator on the Defactify test set** (`Label_B`) — can the model tell each of
   the 5 diffusion models (MidJourney v6, DALL-E 3, SD3, SDXL, SD2.1) apart from the
   real photo of the SAME scene? This is the honest measure of artifact learning.
2. **140k StyleGAN test + DF40 face holdout** — did multi-generator training make the
   face detector worse?
3. **The real-world images** (Gemini, ChatGPT) — v1 vs v2 side by side.

In [ ]:
import torch.nn as nn
from collections import Counter
from tqdm.auto import tqdm

softmax = nn.Softmax(dim=1)

test_b = defactify["test"]["Label_B"]

total = Counter()
correct = Counter()
start = 0
model.eval()
with torch.no_grad():
    for images, labels in tqdm(def_test_loader, desc="per-generator eval"):
        images, labels = images.to(device), labels.to(device)
        preds = softmax(model(images)).argmax(dim=1).cpu()
        for j in range(len(labels)):
            g = test_b[test_ds_def.indices[start + j]]
            total[g] += 1
            correct[g] += int(preds[j].item() == labels[j].item())
        start += len(labels)

rows = [(gen_names[g], n, correct[g] / n) for g, n in total.items()]
for name, n, acc in sorted(rows, key=lambda r: -r[1]):
    print(f"{name:20s} {n:6d}  {acc * 100:6.2f}%")
agg = sum(correct.values()) / sum(total.values())
print(f"{'-' * 38}\n{'ALL':20s} {sum(total.values()):6d}  {agg * 100:6.2f}%")

In [ ]:
# Did multi-generator training hurt the original face tasks?
va_loss, va_acc = evaluate(model, test_loader, criterion, device)
print(f"140k test (StyleGAN vs FFHQ):      accuracy {va_acc:.4f}  (v1 score was 0.9940)")

va_loss2, va_acc2 = evaluate(model, df40_test_loader, criterion, device)
print(f"DF40 face holdout (40 techniques): accuracy {va_acc2:.4f}")

### Re-check the images that failed before

Put any image you want checked into `/content/drive/MyDrive/deepfake_resnet50/test_images/`
(the Gemini boy image and the two ChatGPT images from the earlier check). The cell below
loads both v1 (StyleGAN-only) and v2 (multi-generator) side by side so you can see whether
multi-generator training actually fixed the failure.

In [ ]:
import glob
import os
from PIL import Image

test_images_dir = "/content/drive/MyDrive/deepfake_resnet50/test_images"
os.makedirs(test_images_dir, exist_ok=True)

files = sorted(p for p in glob.glob(os.path.join(test_images_dir, "*"))
               if os.path.splitext(p)[1].lower() in {".jpg", ".jpeg", ".png"})

if not files:
    print("no test images found yet — drop images into the test_images folder and re-run this cell")
else:
    model_v1 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model_v1.fc = nn.Linear(model_v1.fc.in_features, 2)
    model_v1.load_state_dict(torch.load(checkpoint_v1, map_location=device))
    model_v1 = model_v1.to(device).eval()

    print(f"{'file':46s} {'v1 (StyleGAN-only)':20s} {'v2 (multi-generator)':20s}")
    for f in files:
        img = eval_transform(Image.open(f).convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            p1 = softmax(model_v1(img))[0].tolist()
            p2 = softmax(model(img))[0].tolist()
        c1 = "REAL" if p1[0] > p1[1] else "FAKE"
        c2 = "REAL" if p2[0] > p2[1] else "FAKE"
        print(f"{os.path.basename(f):46s} {c1} (r={p1[0]:.2f})          {c2} (r={p2[0]:.2f})")

In [ ]:
# Save the multi-generator model as v2
save_dir = "/content/drive/MyDrive/deepfake_resnet50"
v2_path = os.path.join(save_dir, "deepfake_resnet50_v2.pt")
torch.save(model.state_dict(), v2_path)
print(f"saved: {v2_path} ({os.path.getsize(v2_path)/1e6:.1f} MB)")